In [15]:
# Cell 1: Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Add src to path
import sys
sys.path.append('../src')

from data.dataset_loader import HuggingFaceDatasetLoader
from utils.config import Config

# Setup plotting
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)

print("Setup completed successfully!")

# Cell 2: Load Dataset
# Initialize loader
loader = HuggingFaceDatasetLoader()
config = Config()

# Load dataset
print("Loading HuggingFace dataset: adealvii/Cleaned-Indonesian-Tweet")
df = loader.load_dataset("adealvii/Cleaned-Indonesian-Tweet")

if df is not None:
    print(f"✅ Dataset loaded successfully!")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nFirst 5 rows:")
    display(df.head())
else:
    print("❌ Failed to load dataset")

# Cell 3: Dataset Information
# Get comprehensive dataset info
info = loader.explore_dataset()

print("=== DATASET INFORMATION ===")
for key, value in info.items():
    if isinstance(value, dict):
        print(f"\n{key.upper()}:")
        for k, v in value.items():
            print(f"  {k}: {v}")
    else:
        print(f"{key}: {value}")

# Cell 4: Data Quality Check
print("=== DATA QUALITY ANALYSIS ===")

# Missing values
print("Missing values:")
print(df.isnull().sum())

# Duplicate check
duplicates = df.duplicated().sum()
print(f"\nDuplicate rows: {duplicates}")

# Empty text check
if 'text' in df.columns:
    empty_texts = df['text'].str.len() == 0
    print(f"Empty texts: {empty_texts.sum()}")
elif 'tweet' in df.columns:
    empty_texts = df['tweet'].str.len() == 0
    print(f"Empty tweets: {empty_texts.sum()}")

# Data types
print(f"\nData types:")
print(df.dtypes)

# Cell 5: Text Analysis
# Auto-detect text column
text_col = None
for col in ['text', 'tweet', 'content', 'message']:
    if col in df.columns:
        text_col = col
        break

if text_col:
    print(f"=== TEXT ANALYSIS (Column: {text_col}) ===")
    
    # Text length statistics
    text_lengths = df[text_col].str.len()
    
    print("Text Length Statistics:")
    print(f"  Mean: {text_lengths.mean():.2f}")
    print(f"  Median: {text_lengths.median():.2f}")
    print(f"  Min: {text_lengths.min()}")
    print(f"  Max: {text_lengths.max()}")
    print(f"  Std: {text_lengths.std():.2f}")
    
    # Plot text length distribution
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 3, 1)
    plt.hist(text_lengths, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
    plt.title('Text Length Distribution')
    plt.xlabel('Character Count')
    plt.ylabel('Frequency')
    
    plt.subplot(1, 3, 2)
    plt.boxplot(text_lengths)
    plt.title('Text Length Box Plot')
    plt.ylabel('Character Count')
    
    plt.subplot(1, 3, 3)
    # Word count
    word_counts = df[text_col].str.split().str.len()
    plt.hist(word_counts, bins=30, alpha=0.7, color='lightgreen', edgecolor='black')
    plt.title('Word Count Distribution')
    plt.xlabel('Word Count')
    plt.ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()

# Cell 6: Label Analysis
# Auto-detect label column
label_col = None
for col in ['label', 'sentiment', 'class', 'target']:
    if col in df.columns:
        label_col = col
        break

if label_col:
    print(f"=== LABEL ANALYSIS (Column: {label_col}) ===")
    
    # Label distribution
    label_counts = df[label_col].value_counts()
    print("Label Distribution:")
    print(label_counts)
    
    # Plot label distribution
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 2, 1)
    label_counts.plot(kind='bar', color=['#ff9999', '#66b3ff', '#99ff99'])
    plt.title('Label Distribution')
    plt.xlabel('Labels')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    
    plt.subplot(1, 2, 2)
    plt.pie(label_counts.values, labels=label_counts.index, autopct='%1.1f%%', 
            colors=['#ff9999', '#66b3ff', '#99ff99'])
    plt.title('Label Distribution (Percentage)')
    
    plt.tight_layout()
    plt.show()
    
    # Text length by label
    if text_col:
        plt.figure(figsize=(10, 6))
        for label in df[label_col].unique():
            subset = df[df[label_col] == label][text_col].str.len()
            plt.hist(subset, alpha=0.7, label=f'{label} (n={len(subset)})', bins=30)
        
        plt.title('Text Length Distribution by Label')
        plt.xlabel('Character Count')
        plt.ylabel('Frequency')
        plt.legend()
        plt.show()

# Cell 7: Sample Texts by Label
if text_col and label_col:
    print("=== SAMPLE TEXTS BY LABEL ===")
    
    for label in df[label_col].unique():
        print(f"\n--- {label.upper()} SAMPLES ---")
        samples = df[df[label_col] == label][text_col].head(3)
        for i, text in enumerate(samples, 1):
            print(f"{i}. {text}")

# Cell 8: Word Frequency Analysis
if text_col:
    print("=== WORD FREQUENCY ANALYSIS ===")
    
    # Combine all texts
    all_texts = ' '.join(df[text_col].astype(str))
    
    # Basic word count (split by space)
    words = all_texts.lower().split()
    word_freq = Counter(words)
    
    print(f"Total words: {len(words)}")
    print(f"Unique words: {len(word_freq)}")
    print(f"\nTop 20 most common words:")
    
    top_words = word_freq.most_common(20)
    for word, count in top_words:
        print(f"  {word}: {count}")
    
    # Plot word frequency
    words_df = pd.DataFrame(top_words[:15], columns=['word', 'frequency'])
    
    plt.figure(figsize=(12, 6))
    sns.barplot(data=words_df, x='frequency', y='word', palette='viridis')
    plt.title('Top 15 Most Frequent Words')
    plt.xlabel('Frequency')
    plt.show()

# Cell 9: Word Clouds by Label
if text_col and label_col:
    print("=== WORD CLOUDS BY LABEL ===")
    
    labels = df[label_col].unique()
    n_labels = len(labels)
    
    plt.figure(figsize=(15, 5 * ((n_labels + 1) // 2)))
    
    for i, label in enumerate(labels, 1):
        subset_text = ' '.join(df[df[label_col] == label][text_col].astype(str))
        
        plt.subplot((n_labels + 1) // 2, 2, i)
        
        if len(subset_text.strip()) > 0:
            wordcloud = WordCloud(
                width=400, height=300, 
                background_color='white',
                colormap='viridis',
                max_words=50
            ).generate(subset_text)
            
            plt.imshow(wordcloud, interpolation='bilinear')
            plt.title(f'{label.upper()} Words')
            plt.axis('off')
        else:
            plt.text(0.5, 0.5, f'No data for {label}', 
                    ha='center', va='center', transform=plt.gca().transAxes)
            plt.title(f'{label.upper()} Words')
            plt.axis('off')
    
    plt.tight_layout()
    plt.show()

# Cell 10: Data Preparation
print("=== DATA PREPARATION ===")

# Prepare train/test split
train_df, test_df = loader.prepare_data(
    text_column=text_col,
    label_column=label_col,
    test_size=0.2,
    stratify=True,
    random_state=42
)

print(f"Training data: {len(train_df)} samples")
print(f"Test data: {len(test_df)} samples")

# Check label distribution in splits
if label_col:
    print(f"\nLabel distribution in training set:")
    print(train_df[label_col].value_counts(normalize=True))
    
    print(f"\nLabel distribution in test set:")
    print(test_df[label_col].value_counts(normalize=True))

# Save processed data
train_path, test_path = loader.save_processed_data(train_df, test_df)
print(f"\n✅ Data saved:")
print(f"  Training: {train_path}")
print(f"  Test: {test_path}")

# Cell 11: Summary Report
print("=== DATASET SUMMARY REPORT ===")
print(f"Dataset: adealvii/Cleaned-Indonesian-Tweet")
print(f"Total samples: {len(df):,}")
print(f"Text column: {text_col}")
print(f"Label column: {label_col}")

if text_col:
    print(f"Average text length: {df[text_col].str.len().mean():.1f} characters")
    print(f"Average word count: {df[text_col].str.split().str.len().mean():.1f} words")

if label_col:
    print(f"Number of classes: {df[label_col].nunique()}")
    print(f"Classes: {list(df[label_col].unique())}")

print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")

print(f"\n✅ Dataset exploration completed!")
print(f"Ready for preprocessing and model training.")

Setup completed successfully!
Loading HuggingFace dataset: adealvii/Cleaned-Indonesian-Tweet


ERROR:root:Error loading dataset: Dataset 'cahya/sentiment-bahasa' doesn't exist on the Hub or cannot be accessed.


❌ Failed to load dataset


ValueError: Dataset belum di-load. Jalankan load_dataset() terlebih dahulu.